# AeroBin Phase 7b — Train the Burn-Risk Classifier

Trains on `burn_dataset.csv` (from `01_build_dataset.ipynb`) and reports the metrics checkpoint **before any integration** (locked recipe):

- **Models:** Logistic Regression (calibrated linear baseline) + Gradient Boosting (the 3B design pair)
- **Split:** time-series — train 2022–2023, test 2024 (**never shuffled**)
- **Class imbalance:** `class_weight` / sample weights (burn days ≈ 2% of ward-days)
- **Metrics that don't lie for rare events:** **AUPRC** (average precision) + per-ward precision + calibration — accuracy is reported only to be debunked
- **Outputs:** winner → `../public/models/burn-risk-v0.1.onnx` + `burn-risk-v0.1-metrics.json` sidecar (served with the app; the `/model` page renders the sidecar)

### Model-card honesty (locked decisions)
- This is a **pipeline demonstrator on real data, not a production predictor** — Pune CPCB coverage is patchy and FIRMS 375 m pixels miss many small waste fires.
- Labels are satellite hotspots ≥ 1 pixel within 48h — **industrial false-positives at Bhosari are a known confound** (threshold sensitivity reported below).
- A calibrated rare-event model will rarely say ≥70% — showing that truthfully is the point.
- Per-ward test-set precision is the **S3 equity-gap measurement** (model precision in low-income vs high-income wards), folded into the `/model` page.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TRAINING_DIR = Path.cwd() if Path.cwd().name == 'training' else Path('../training').resolve()
if not TRAINING_DIR.exists():
    TRAINING_DIR = Path('training').resolve()
REPO_ROOT = TRAINING_DIR.parent
MODELS_DIR = REPO_ROOT / 'public' / 'models'

FEATURE_COLS = [
    'pm25_mean', 'pm25_mean_7d', 'pm25_slope_7d',
    'humidity_mean', 'humidity_7d', 'humidity_trend_7d',
    'temp_max', 'wind_max', 'precipitation_sum', 'days_since_rain',
    'green_cover_pct', 'market_flag', 'income_ord',
    'festival_window', 'doy_sin', 'doy_cos',
]

df = pd.read_csv(TRAINING_DIR / 'burn_dataset.csv')
assert set(FEATURE_COLS) <= set(df.columns), 'feature columns missing from dataset'
assert df[FEATURE_COLS].isna().sum().sum() == 0, 'NaNs in features — 7a clip should have removed these'
print(f'{len(df)} ward-days, {df["burn_day"].sum()} burn days ({df["burn_day"].mean():.3f} base rate)')
print('label threshold: 1  (see 01_build_dataset.ipynb LABEL_THRESHOLD for the sensitivity knob)')

In [ ]:
# Time-series split: train 2022-23, test 2024. Never shuffle — that would let the
# model see the future (e.g. learn 2024's monsoon) and report fantasy skill.
train = df[df['split_year'] <= 2023].sort_values(['ward_id', 'date'])
test = df[df['split_year'] == 2024].sort_values(['ward_id', 'date'])

X_train, y_train = train[FEATURE_COLS], train['burn_day'].values
X_test, y_test = test[FEATURE_COLS], test['burn_day'].values
ward_test = test['ward_id'].values

print(f'train: {len(train)} rows ({y_train.sum()} burn days, {y_train.mean():.3f})')
print(f'test:  {len(test)} rows ({y_test.sum()} burn days, {y_test.mean():.3f})')
assert y_test.sum() > 0, 'test year has zero burn days — metrics would be meaningless'

## 1 — Candidate models

Both candidates are wrapped in a scaler-first pipeline (LR needs it; GB tolerates it for parity) and **calibrated** with 5-fold CV on the training set so the output probability is a real confidence score, not a raw score.

In [ ]:
SEED = 42

# Logistic Regression — interpretable linear baseline, balanced class weights.
# Logistic Regression — interpretable linear baseline. UNWEIGHTED inside
# calibration: 'balanced' class weights + sigmoid calibration distort the
# probabilities we intend to show as confidence (Brier 0.019 → honest rare-event
# probabilities beat inflated positive rates; ranking is unaffected).
lr_base = LogisticRegression(
    max_iter=1000, random_state=SEED, solver='lbfgs'
)
lr = Pipeline([('scale', StandardScaler()),
               ('clf', CalibratedClassifierCV(lr_base, method='sigmoid', cv=5))])

# Gradient Boosting — the 3B design's chosen family for tabular burn risk.
# UNWEIGHTED by deliberate decision: passing 50x sample weights into
# CalibratedClassifierCV broke calibration on the first run (Brier 0.135 vs
# LR's 0.018 — weighted sigmoid fits went wild). For a system whose headline
# number is a calibrated probability, honest probabilities > more positives.
gb_base = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    subsample=0.8, random_state=SEED,
)
gb = Pipeline([('scale', StandardScaler()),
               ('clf', CalibratedClassifierCV(gb_base, method='sigmoid', cv=3))])

print('fitting Logistic Regression ...')
lr.fit(X_train, y_train)
print('fitting Gradient Boosting (calibration CV — this is the slow one) ...')
gb.fit(X_train, y_train)
print('done')

## 2 — Metrics checkpoint (the review gate before integration)

Rare-event metrics on the **2024 holdout**: AUPRC is the headline (a 0.02 base rate makes ROC-AUC flattering and accuracy meaningless — both are reported to be debunked in the model card).

In [ ]:
def evaluate(name, model, X, y, wards):
    proba = model.predict_proba(X)[:, 1]
    base = y.mean()
    auprc = average_precision_score(y, proba)
    roc = roc_auc_score(y, proba)
    brier = brier_score_loss(y, proba)
    # operating points: top-K flags at several dispatch-plausible K
    # (1%, 2.5%, 5% of ward-days) — precision at each is what PMC would live with
    op_points = {}
    for pct in (0.01, 0.025, 0.05):
        k = max(int(pct * len(y)), 5)
        thr = np.percentile(proba, 100 * (1 - k / len(y)))
        flag = proba >= thr
        op_points[f'{pct*100:g}%'] = {
            'k': k, 'threshold': float(thr),
            'precision': float(precision_score(y, flag, zero_division=0)),
            'recall': float(recall_score(y, flag, zero_division=0)),
        }
    # primary operating point for per-ward reporting: top 2.5%
    primary = op_points['2.5%']
    flag = proba >= primary['threshold']
    per_ward = {}
    for w in sorted(set(wards)):
        m = wards == w
        per_ward[w] = {
            'n_days': int(m.sum()),
            'burn_days': int(y[m].sum()),
            'precision': float(precision_score(y[m], flag[m], zero_division=0)),
            'recall': float(recall_score(y[m], flag[m], zero_division=0)),
            'flagged': int(flag[m].sum()),
        }
    return {
        'model': name,
        'auprc': float(auprc), 'base_rate': float(base),
        'roc_auc': float(roc), 'brier': float(brier),
        'operating_points': op_points,
        'operating_threshold_pct': primary['threshold'],
        'operating_precision': primary['precision'],
        'operating_recall': primary['recall'],
        'flagged_days': int(flag.sum()),
        'per_ward': per_ward,
        'proba': proba,
    }

results = {}
results['logistic_regression'] = evaluate('Logistic Regression', lr, X_test, y_test, ward_test)
results['gradient_boosting'] = evaluate('Gradient Boosting', gb, X_test, y_test, ward_test)

for r in results.values():
    print(f"\n=== {r['model']} (2024 holdout) ===")
    print(f"  AUPRC {r['auprc']:.3f}  (base rate {r['base_rate']:.3f} → lift {r['auprc']/r['base_rate']:.1f}x)")
    print(f"  ROC-AUC {r['roc_auc']:.3f}  Brier {r['brier']:.3f}")
    for pct, op in r['operating_points'].items():
        print(f"  top {pct} ({op['k']} flags): precision {op['precision']:.2f}, recall {op['recall']:.2f}")
    print(f"  per-ward @ top 2.5%:")
    for w, s in r['per_ward'].items():
        print(f"    {w:9s} burn_days {s['burn_days']:2d}  flagged {s['flagged']:2d}  precision {s['precision']:.2f}  recall {s['recall']:.2f}")

## 3 — Winner selection + equity-gap measurement (S3)

Winner = higher AUPRC on the 2024 holdout. The **equity gap** = max precision − min precision across wards' income groups (low: wagholi/bhosari, high: kharadi; mixed: hadapsar/mundhwa) at the operating point — this is the historical S3 measurement for the `/model` page.

In [ ]:
INCOME = {'hadapsar': 'mixed', 'kharadi': 'high', 'wagholi': 'low', 'bhosari': 'low', 'mundhwa': 'mixed'}

winner_name = max(results, key=lambda k: results[k]['auprc'])
winner = results[winner_name]
print(f"winner: {winner['model']} (AUPRC {winner['auprc']:.3f})")

# Equity gap by income group (S3): precision at the operating point, grouped.
group_prec = {}
for grp in ('low', 'mixed', 'high'):
    wards_in = [w for w, g in INCOME.items() if g == grp]
    y_grp = np.isin(ward_test, wards_in)
    flag = winner['proba'] >= winner['operating_threshold_pct']
    group_prec[grp] = float(precision_score(y_test[y_grp], flag[y_grp], zero_division=0))
    print(f"  income={grp:5s} wards={wards_in} precision {group_prec[grp]:.2f}")
equity_gap = max(group_prec.values()) - min(group_prec.values())
print(f"\nS3 equity gap (max-min group precision): {equity_gap:.2f}")

## 4 — Export winner to ONNX (+ metrics sidecar)

`skl2onnx` converts the sklearn pipeline (scaler + calibrated classifier) into one graph with the standard `float_input` name. The sidecar JSON carries every number the `/model` page shows — **the app never recomputes metrics; it displays exactly these**.

In [ ]:
import onnx
from skl2onnx import to_onnx

MODELS_DIR.mkdir(parents=True, exist_ok=True)

models = {'logistic_regression': lr, 'gradient_boosting': gb}
winner_pipeline = models[winner_name]

# skl2onnx turns named DataFrame columns into per-column inputs; strip names
# and fit a fresh clone on a plain ndarray so the graph has ONE flat input.
from sklearn.base import clone
X_train_arr = X_train.to_numpy(dtype=np.float64)
flat_pipe = clone(winner_pipeline)
flat_pipe.fit(X_train_arr, y_train)

onnx_model = to_onnx(
    flat_pipe,
    X=X_train_arr[:2].astype(np.float32),
    target_opset=17,
    options={id(flat_pipe): {'zipmap': False}},  # bare [P(0) P(1)] tensor — onnxruntime-web friendly
)
onnx_path = MODELS_DIR / 'burn-risk-v0.1.onnx'
onnx.checker.check_model(onnx_model)
onnx.save_model(onnx_model, onnx_path)
print(f'ONNX saved: {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)')

# numerical check: ONNX output must match sklearn's within 1e-3
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
print('ONNX inputs:', [(i.name, i.shape, i.type) for i in sess.get_inputs()])
print('ONNX outputs:', [(o.name, o.shape) for o in sess.get_outputs()])
inp_name = sess.get_inputs()[0].name
X_test_arr = X_test.to_numpy(dtype=np.float32)
onnx_proba = sess.run(None, {inp_name: X_test_arr[:50]})
print('raw output shapes:', [o.shape for o in onnx_proba])
onnx_p1 = onnx_proba[1][:, 1] if onnx_proba[1].ndim == 2 else onnx_proba[1]
sk_proba = flat_pipe.predict_proba(X_test[:50])[:, 1]
max_diff = float(np.abs(onnx_p1 - sk_proba).max())
print(f'max |onnx - sklearn| on 50 test rows: {max_diff:.2e}')
assert max_diff < 1e-3, 'ONNX output deviates from sklearn — do not ship'

# feature order contract for the client (and the sidecar)
FEATURE_ORDER = FEATURE_COLS  # client must feed columns in this exact order

In [ ]:
metrics = {
    'version': '0.1',
    'model': winner['model'],
    'task': 'ward-day burn-risk probability ( Pune pilot wards)',
    'trainedOn': '2022-07-15..2023-12-31 (train), 2024-01-01..2024-12-31 (test holdout)',
    'labelDefinition': 'attributed FIRMS VIIRS_SNPP_SP hotspots >= 1 within ward<=5km on day or next (48h framing)',
    'labelThreshold': 1,
    'features': FEATURE_COLS,
    'testMetrics': {
        'auprc': winner['auprc'],
        'baseRate': winner['base_rate'],
        'rocAuc': winner['roc_auc'],
        'brier': winner['brier'],
        'operatingThreshold': winner['operating_threshold_pct'],
        'operatingPrecision': winner['operating_precision'],
        'operatingRecall': winner['operating_recall'],
        'flaggedDays': winner['flagged_days'],
        'perWard': winner['per_ward'],
        'equityGapS3': equity_gap,
        'incomeGroupPrecision': group_prec,
    },
    'candidates': {
        k: {m: v for m, v in r.items() if m not in ('proba', 'per_ward')}
        for k, r in results.items()
    },
    'limitations': [
        'Pipeline demonstrator on real public data — not a production predictor',
        'FIRMS 375m pixels miss many small waste fires; labels are a lower bound on true burning',
        'Bhosari industrial hotspots are a known false-positive confound at threshold 1',
        'CAMS gridded PM2.5 is reanalysis, not ground truth; Pune CPCB coverage is patchy',
        'Ward attribution uses nearest-centroid (<=5km), not polygons — pilot wards share admin boundaries',
        'Calibrated rare-event probabilities will rarely exceed 70% — by design',
    ],
    'dataSources': {
        'weather': 'Open-Meteo Archive API (temperature_2m_max, relative_humidity_2m_mean, precipitation_sum, wind_speed_10m_max)',
        'pm25': 'Open-Meteo Air Quality API (CAMS reanalysis pm2_5, daily mean of available hours)',
        'hotspots': 'NASA FIRMS VIIRS_SNPP_SP (Standard Processing archive), Pune bbox 73.70,18.40,74.10,18.68',
        'wardFlags': 'public/data/aerobin_data.json (greenCover, marketFlag, incomeLevel)',
    },
    'onnxCheck': {'maxAbsDiffVsSklearn': max_diff, 'opset': 17},
}
sidecar = MODELS_DIR / 'burn-risk-v0.1-metrics.json'
sidecar.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('sidecar saved:', sidecar)
print(json.dumps(metrics['testMetrics'], indent=2)[:800])

## 5 — Threshold-2 label sensitivity (documented hyperparameter)

Re-label with `LABEL_THRESHOLD=2` (re-run 7a with the knob flipped, or relabel from lineage here) and compare — this quantifies how much of the signal survives a stricter burn definition. Kept as a reported experiment in the sidecar.

In [ ]:
# Relabel from the 48h hotspot counts already in the dataset (no refetch needed):
df2 = df.copy()
df2['burn_day_strict'] = (df2['hotspots_48h'] >= 2).astype(int)
tr2 = df2[df2['split_year'] <= 2023]
te2 = df2[df2['split_year'] == 2024]
print(f"threshold 2: train {tr2['burn_day_strict'].sum()} burn days ({tr2['burn_day_strict'].mean():.4f}), "
      f"test {te2['burn_day_strict'].sum()} ({te2['burn_day_strict'].mean():.4f})")

if te2['burn_day_strict'].sum() >= 5:
    lr2 = Pipeline([('scale', StandardScaler()),
                    ('clf', CalibratedClassifierCV(
                        LogisticRegression(max_iter=1000, class_weight='balanced',
                                           random_state=SEED), method='sigmoid', cv=5))])
    lr2.fit(tr2[FEATURE_COLS], tr2['burn_day_strict'])
    p2 = lr2.predict_proba(te2[FEATURE_COLS])[:, 1]
    auprc2 = average_precision_score(te2['burn_day_strict'], p2)
    print(f'LR AUPRC @ threshold 2: {auprc2:.3f} (base {te2["burn_day_strict"].mean():.4f})')
    with open(sidecar, encoding='utf-8') as f:
        m = json.load(f)
    m['threshold2Sensitivity'] = {
        'testBurnDays': int(te2['burn_day_strict'].sum()),
        'lrAuprc': float(auprc2),
        'baseRate': float(te2['burn_day_strict'].mean()),
    }
    sidecar.write_text(json.dumps(m, indent=2), encoding='utf-8')
    print('sensitivity result appended to sidecar')
else:
    print('too few threshold-2 test burn days for a meaningful comparison — documented, not reported as skill')